In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
base_path = "/content/drive/MyDrive/textile_dataset"

In [3]:
import cv2
import os

def process_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for img_name in os.listdir(input_folder):
        img_path = os.path.join(input_folder, img_name)
        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.resize(img, (64, 64))
        cv2.imwrite(os.path.join(output_folder, img_name), img)

# Create processed dataset
process_folder(base_path + "/train/normal", "/content/processed/train/normal")
process_folder(base_path + "/train/defect", "/content/processed/train/defect")
process_folder(base_path + "/test/normal", "/content/processed/test/normal")
process_folder(base_path + "/test/defect", "/content/processed/test/defect")

In [4]:
import shutil
os.makedirs("/content/gan_dataset/defects", exist_ok=True)

source = "/content/processed/train/defect"
dest = "/content/gan_dataset/defects"

for file in os.listdir(source):
    shutil.copy(os.path.join(source, file), dest)

In [5]:
import os
print(len(os.listdir("/content/gan_dataset/defects")))

176


In [6]:
# =========================
# 1. IMPORTS
# =========================
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import os

# =========================
# 2. CONFIG
# =========================
image_size = 64
batch_size = 32
latent_dim = 100
num_epochs = 30
lr = 0.0002

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 3. DATASET PATH
# =========================
dataset_path = "/content/gan_dataset"

# =========================
# 4. DATA LOADING
# =========================
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

dataset = ImageFolder(root=dataset_path, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print("Dataset loaded:", len(dataset))

# =========================
# 5. GENERATOR
# =========================
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, 4, 1, 0),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

# =========================
# 6. DISCRIMINATOR (FIXED)
# =========================
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),   # 64 → 32
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1), # 32 → 16
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Conv2d(128, 256, 4, 2, 1), # 16 → 8
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            nn.Conv2d(256, 1, 4, 1, 0),  # 8 → 1
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

# =========================
# 7. INITIALIZE
# =========================
G = Generator().to(device)
D = Discriminator().to(device)

criterion = nn.BCELoss()

opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

# =========================
# 8. TRAINING LOOP (FIXED)
# =========================
print("Starting GAN training...")

for epoch in range(num_epochs):
    for real, _ in dataloader:

        real = real.to(device)
        batch_size = real.size(0)

        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # ---- Train Discriminator ----
        opt_D.zero_grad()

        out_real = D(real)
        out_real = out_real.view(out_real.size(0), -1).mean(dim=1, keepdim=True)
        loss_real = criterion(out_real, real_labels)

        noise = torch.randn(batch_size, latent_dim, 1, 1).to(device)
        fake = G(noise)

        out_fake = D(fake.detach())
        out_fake = out_fake.view(out_fake.size(0), -1).mean(dim=1, keepdim=True)
        loss_fake = criterion(out_fake, fake_labels)

        loss_D = loss_real + loss_fake
        loss_D.backward()
        opt_D.step()

        # ---- Train Generator ----
        opt_G.zero_grad()

        out_fake = D(fake)
        out_fake = out_fake.view(out_fake.size(0), -1).mean(dim=1, keepdim=True)
        loss_G = criterion(out_fake, real_labels)

        loss_G.backward()
        opt_G.step()

    print(f"Epoch [{epoch+1}/{num_epochs}]  D Loss: {loss_D.item():.4f}, G Loss: {loss_G.item():.4f}")

# =========================
# 9. GENERATE IMAGES
# =========================
os.makedirs("/content/generated_defects", exist_ok=True)

G.eval()
with torch.no_grad():
    noise = torch.randn(30, latent_dim, 1, 1).to(device)
    fake_images = G(noise)

    for i, img in enumerate(fake_images):
        torchvision.utils.save_image(img, f"/content/generated_defects/img_{i}.png", normalize=True)

print("✅ Generated images saved in /content/generated_defects/")

Dataset loaded: 176
Starting GAN training...
Epoch [1/30]  D Loss: 0.6322, G Loss: 2.9347
Epoch [2/30]  D Loss: 0.3871, G Loss: 3.6304
Epoch [3/30]  D Loss: 0.1900, G Loss: 4.0855
Epoch [4/30]  D Loss: 0.1840, G Loss: 4.4215
Epoch [5/30]  D Loss: 0.1033, G Loss: 4.6937
Epoch [6/30]  D Loss: 0.1108, G Loss: 4.9723
Epoch [7/30]  D Loss: 0.0768, G Loss: 5.1727
Epoch [8/30]  D Loss: 0.0570, G Loss: 5.3666
Epoch [9/30]  D Loss: 0.0585, G Loss: 5.4483
Epoch [10/30]  D Loss: 0.0617, G Loss: 5.7325
Epoch [11/30]  D Loss: 0.0307, G Loss: 5.7135
Epoch [12/30]  D Loss: 0.0450, G Loss: 6.0047
Epoch [13/30]  D Loss: 0.0545, G Loss: 6.1500
Epoch [14/30]  D Loss: 0.0249, G Loss: 6.1818
Epoch [15/30]  D Loss: 0.0277, G Loss: 5.8866
Epoch [16/30]  D Loss: 0.0208, G Loss: 6.2207
Epoch [17/30]  D Loss: 0.0253, G Loss: 5.8899
Epoch [18/30]  D Loss: 0.0475, G Loss: 5.2837
Epoch [19/30]  D Loss: 0.0469, G Loss: 5.6068
Epoch [20/30]  D Loss: 0.0227, G Loss: 5.6937
Epoch [21/30]  D Loss: 0.0270, G Loss: 5.432